# Sprint 1.2 — Ingestion & Vectorisation : Pipeline RAG de bout en bout

**Objectif** : Construire un RAG complet sur les documents du logement.

**Sujets couverts** :
- Extraction de texte PDF (PyMuPDF)
- 3 stratégies de chunking comparées
- Embeddings multilingues
- FAISS vs ChromaDB
- Pipeline RAG LangChain complet
- Évaluation sur 20 questions étalons

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# Vérification des données
docs_dir = "../data/documents"
pdfs = [f for f in os.listdir(docs_dir) if f.endswith('.pdf')]
print(f"PDFs disponibles : {len(pdfs)}")
for f in pdfs:
    size = os.path.getsize(os.path.join(docs_dir, f)) / 1024
    print(f"  - {f} ({size:.0f} Ko)")

## 1. Extraction de texte avec PyMuPDF

**Note** : `pymupdf` s'installe via `pip install pymupdf` mais s'importe `import fitz` — source de confusion !

In [ ]:
import fitz  # pymupdf

# Charger la notice chaudière
path = "../data/documents/notice_chaudiere.pdf"
doc = fitz.open(path)

print(f"Nombre de pages : {len(doc)}")
print(f"\n=== Page 1 (extrait) ===")
page_text = doc[0].get_text()
print(page_text[:500] + "...")
doc.close()

## 2. Comparaison de 3 stratégies de chunking

Nous allons tester sur le même document les 3 approches :
1. **Taille fixe** (512 caractères) — simple mais coupe les idées
2. **Récursif par séparateurs** — intelligent, préserve la structure
3. **Sémantique** — le plus précis, le plus lent

In [ ]:
from homebutler.rag.ingestion import (
    load_pdf_with_metadata, chunk_fixed_size, chunk_recursive
)

# Charger tous les documents
from homebutler.rag.ingestion import ingest_all_documents

# Stratégie 1 : taille fixe
docs_pages = [d for f in os.listdir('../data/documents') if f.endswith('.pdf')
              for d in load_pdf_with_metadata(f'../data/documents/{f}')]

chunks_fixed = chunk_fixed_size(docs_pages, chunk_size=512, chunk_overlap=50)
chunks_recursive = chunk_recursive(docs_pages, chunk_size=512, chunk_overlap=50)

print("=== Comparaison des stratégies ===")
print(f"Stratégie 1 (taille fixe)     : {len(chunks_fixed)} chunks")
print(f"Stratégie 2 (récursif)        : {len(chunks_recursive)} chunks")

In [ ]:
import numpy as np

def analyze_chunks(chunks, label):
    sizes = [len(c.page_content) for c in chunks]
    print(f"\n{label}")
    print(f"  Nombre de chunks : {len(chunks)}")
    print(f"  Taille moy  : {np.mean(sizes):.0f} chars")
    print(f"  Taille min  : {min(sizes)} chars")
    print(f"  Taille max  : {max(sizes)} chars")
    print(f"  Exemple : {chunks[10].page_content[:150]}...")

analyze_chunks(chunks_fixed, "Stratégie 1 — Taille fixe")
analyze_chunks(chunks_recursive, "Stratégie 2 — Récursif")

## 3. Génération d'embeddings

Modèle utilisé : `paraphrase-multilingual-MiniLM-L12-v2`
- 50 langues supportées
- Dimension : 384
- Taille : ~120 Mo
- Tourne en CPU

In [ ]:
from homebutler.rag.vectorstore import get_embeddings

embeddings = get_embeddings()

# Test
test_queries = [
    "température chaudière nuit",
    "chauffe-eau réglage nocturne",  # synonyme
    "boiler night temperature setting",  # anglais — le modèle est multilingue !
]

vecs = embeddings.embed_documents(test_queries)
print(f"Dimension des embeddings : {len(vecs[0])}")

# Similarité cosinus
import numpy as np
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\nSimilarité 'temp chaudière' vs 'chauffe-eau' : {cosine_sim(vecs[0], vecs[1]):.3f}")
print(f"Similarité 'temp chaudière' vs 'boiler EN'  : {cosine_sim(vecs[0], vecs[2]):.3f}")

## 4. FAISS vs ChromaDB

| | FAISS | ChromaDB |
|---|---|---|
| **Usage** | Docs statiques (notices, bail) | FAQ dynamiques |
| **Stockage** | Fichier binaire | SQLite/JSON persistant |
| **Filtres** | Non (natif) | Oui (métadonnées) |
| **Performance** | Très rapide | Rapide |
| **Mise à jour** | Reconstruire l'index | Insertion dynamique |

In [ ]:
import time
from homebutler.rag.vectorstore import build_faiss_index, build_chroma_db

# Construction FAISS
t0 = time.time()
faiss_store = build_faiss_index(chunks_recursive, save_path="../data/faiss_index", force_rebuild=True)
t_faiss = time.time() - t0
print(f"FAISS construit en {t_faiss:.1f}s")

# Construction ChromaDB
t0 = time.time()
chroma_store = build_chroma_db(chunks_recursive, persist_dir="../data/chroma_db")
t_chroma = time.time() - t0
print(f"ChromaDB construit en {t_chroma:.1f}s")

In [ ]:
# Test de recherche comparative
query = "température recommandée chaudière la nuit"

# FAISS
t0 = time.time()
faiss_results = faiss_store.similarity_search(query, k=3)
t_faiss_search = time.time() - t0

# ChromaDB
t0 = time.time()
chroma_results = chroma_store.similarity_search(query, k=3)
t_chroma_search = time.time() - t0

print(f"FAISS  : {t_faiss_search*1000:.1f}ms | ChromaDB : {t_chroma_search*1000:.1f}ms")
print(f"\n=== Résultats FAISS ===")
for i, doc in enumerate(faiss_results):
    print(f"[{doc.metadata['source']}] {doc.page_content[:150]}...")

## 5. Pipeline RAG LangChain complet

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from homebutler.llm.provider import get_llm
from homebutler.llm.prompts import RAG_QA_TEMPLATE

# Retriever
retriever = faiss_store.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20})

# LLM
llm = get_llm(temperature=0.1)

# Chain
def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_QA_TEMPLATE
    | llm
    | StrOutputParser()
)

# Test
question = "Quelle est la température recommandée pour ma chaudière la nuit ?"
print(f"Question : {question}")
print("\nRéponse HomeButler :")
response = rag_chain.invoke(question)
print(response)

## 6. Évaluation sur 20 questions étalons

Mesure du rappel : combien de questions obtiennent un document pertinent dans le top-3 ?

In [ ]:
BENCHMARK_QUESTIONS = [
    ("Température chaudière nuit", "notice_chaudiere.pdf"),
    ("Code erreur F4 chaudière", "notice_chaudiere.pdf"),
    ("Purge radiateurs procédure", "notice_chaudiere.pdf"),
    ("Entretien annuel chaudière obligation", "notice_chaudiere.pdf"),
    ("Filtre VMC nettoyage fréquence", "notice_vmc.pdf"),
    ("VMC vitesse modes débit", "notice_vmc.pdf"),
    ("Lave-linge erreur E18", "notice_lave_linge.pdf"),
    ("Lave-linge filtre vidange nettoyage", "notice_lave_linge.pdf"),
    ("Préavis location zone tendue", "bail_location.pdf"),
    ("Charges locatives composition", "bail_location.pdf"),
    ("Dépôt garantie caution délai restitution", "bail_location.pdf"),
    ("Réparations locatives obligation locataire", "bail_location.pdf"),
    ("Horaires bruit interdit copropriété", "reglement_copropriete.pdf"),
    ("Animaux compagnie règlement", "reglement_copropriete.pdf"),
    ("Tri sélectif poubelles résidence", "reglement_copropriete.pdf"),
    ("DPE classe D consommation", "dpe.pdf"),
    ("Travaux recommandés isolation fenêtres", "dpe.pdf"),
    ("Aides MaPrimeRenov rénovation", "dpe.pdf"),
    ("Chaudière pression circuit hydraulique", "notice_chaudiere.pdf"),
    ("VMC mode boost cuisine", "notice_vmc.pdf"),
]

hits = 0
results = []
for question, expected_source in BENCHMARK_QUESTIONS:
    docs = retriever.invoke(question)
    sources = [d.metadata.get('source', '') for d in docs]
    hit = any(expected_source in s for s in sources)
    hits += int(hit)
    results.append({
        "Question": question[:50],
        "Source attendue": expected_source,
        "Trouvée": "✅" if hit else "❌",
        "Sources retournées": str(sources[:2]),
    })

import pandas as pd
df_eval = pd.DataFrame(results)
print(df_eval.to_string())
print(f"\n=== Rappel@3 : {hits}/{len(BENCHMARK_QUESTIONS)} = {hits/len(BENCHMARK_QUESTIONS)*100:.0f}% ===")